# 03 — Models & Tuning · thành viên D

**Trần Đình Bảo Quốc — 3124411252**. Thực hiện tuần tự theo `PHAN_CONG.md`: môi trường/baseline A → kNN/SVM/ExtraTrees → boosting/MLP → tuning → v2 → bàn giao E.

Tất cả 32 cấu hình đã được huấn luyện thực tế bằng CLI từng bước, với OOF/test/CV lưu trên đĩa. Notebook mặc định **đọc lại các kết quả này và kiểm tra lại**, không nhận là một lượt huấn luyện mới. Đổi `RECOMPUTE=True` để chạy lại toàn bộ baseline và 32 cấu hình từ đầu. Nếu thiếu artifact, từng bước sẽ tự huấn luyện phần thiếu. Không dùng dữ liệu ngoài hoặc leaderboard để tuning.

Chạy bằng `.venv` Python 3.12 sau khi cài `code/requirements_D.txt`. Dữ liệu tên chuẩn nằm trong `data/` (bản sao nguyên byte từ file gốc tên có `(1)`/`(2)`).

In [1]:
from pathlib import Path
import os, sys, json, hashlib, importlib, importlib.metadata, subprocess
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
ROOT = Path.cwd().resolve()
if not (ROOT / 'PHAN_CONG.md').exists():
    ROOT = ROOT.parent
assert (ROOT / 'PHAN_CONG.md').exists()
sys.path.insert(0, str(ROOT / 'code'))
RECOMPUTE = False  # True = thực sự huấn luyện lại mọi cấu hình, cần thêm thời gian
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import common
experiments = importlib.import_module('03_models_tuning_D')
audit_module = importlib.import_module('03_audit_D')
report = importlib.import_module('03_report_D')
assert common.SEED == 42 and common.N_FOLDS == 5 and common.PRIMARY_METRIC == 'accuracy'
manifest_path = ROOT / 'evidence/models_D/handoff_manifest_D.json'
if manifest_path.exists() and not RECOMPUTE:
    manifest = json.loads(manifest_path.read_text())
    for relative, expected in {**manifest['dataset_sha256'], **manifest['source_sha256']}.items():
        assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, f'{relative} changed; set RECOMPUTE=True'
    for name, expected in manifest['packages'].items():
        assert importlib.metadata.version(name) == expected, f'Install requirements_D.txt: {name}'
print('Python:', sys.executable)
print('Chế độ:', 'HUẤN LUYỆN LẠI' if RECOMPUTE else 'ĐỌC CACHE + AUDIT; huấn luyện nếu thiếu')
display(pd.read_csv(ROOT / 'data/train.csv').shape)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


Python: D:\Documents\Machine_Learning\MLGroup\Music Genre Classification\.venv\Scripts\python.exe
Chế độ: ĐỌC CACHE + AUDIT; huấn luyện nếu thiếu


(14396, 18)

CompletedProcess(args=['D:\\Documents\\Machine_Learning\\MLGroup\\Music Genre Classification\\.venv\\Scripts\\python.exe', '-m', 'pip', 'check'], returncode=0)

## 1. Tái lập baseline A

Giữ nguyên mã A và cô lập đường dẫn đầu ra. HistGB v1 tái lập đạt 0,558488, khớp mốc làm tròn 0,5585. Baseline A là lượt kiểm chứng mã cũ; các mô hình D dùng tiền xử lý trong fold.

In [2]:
baseline_csv = ROOT / 'evidence/baseline_reproduction_D/comparison.csv'
if RECOMPUTE or not baseline_csv.exists():
    subprocess.run([sys.executable, '-u', str(ROOT / 'code/03_reproduce_baseline_D.py')], cwd=ROOT, check=True)
display(pd.read_csv(baseline_csv))
display(pd.read_json(ROOT / 'evidence/baseline_reproduction_D/environment.json', typ='series'))


,model,cv_accuracy,cv_std,macro_f1,seconds
0,histgb_v1,0.558488,0.007060,0.606778,10.983897
1,rf_v1,0.534801,0.007302,0.588839,15.397819
2,logreg_v1,0.525840,0.003620,0.558662,5.890349


python                                                3.12.10
platform                            Windows-11-10.0.26300-SP0
packages    {'numpy': '2.5.4', 'pandas': '3.1.1', 'scikit-...
dtype: object

## 2. Baseline bổ sung trên v1

kNN và SVM dùng StandardScaler trong Pipeline. Median chỉ lấy từ outer train; tần suất không nhãn cố định trên train+test theo quy tắc nhóm. Giữ nguyên 5-fold, seed và thứ tự dòng.

In [3]:
experiments.baselines(resume=not RECOMPUTE)
display(audit_module.audit_all().query("model.str.endswith('v1_base_D')", engine='python'))


Archived result: knn_v1_base_D


Archived result: svm_v1_base_D


Archived result: extratrees_v1_base_D


PASS: 32 models; probabilities, row counts, class order, all 5-fold metrics


,model,family,feature_version,target_encoding,cv_accuracy,cv_std,macro_f1,seconds,audit
9,xgboost_v1_base_D,xgboost,v1,False,0.575159,0.004627,0.615444,24.804181,PASS
13,catboost_v1_base_D,catboost,v1,False,0.572242,0.005501,0.606919,57.152118,PASS
19,lightgbm_v1_base_D,lightgbm,v1,False,0.547652,0.003057,0.607259,27.014440,PASS
20,mlp_v1_base_D,mlp,v1,False,0.547513,0.002681,0.569618,25.947606,PASS
24,svm_v1_base_D,svm,v1,False,0.546124,0.005852,0.579793,148.146806,PASS
30,extratrees_v1_base_D,extratrees,v1,False,0.522854,0.010142,0.572113,15.241423,PASS
31,knn_v1_base_D,knn,v1,False,0.475479,0.007390,0.526504,2.977038,PASS


## 3. Ba boosting và MLP nhỏ trên v1

Không dùng class weights. Boosting dùng số vòng cố định. MLP NumPy có dropout, AdamW và inner holdout 10% để early stop; không sử dụng outer validation để chọn epoch.

In [4]:
experiments.boosting(resume=not RECOMPUTE)
for script in ['03_test_protocol_D.py', '03_test_mlp_D.py']:
    subprocess.run([sys.executable, str(ROOT / 'code' / script)], cwd=ROOT, check=True)


Archived result: lightgbm_v1_base_D


Archived result: xgboost_v1_base_D


Archived result: catboost_v1_base_D


Archived result: mlp_v1_base_D


## 4. Tuning với Optuna

RandomSampler seed 42, ba cấu hình mới/họ + baseline; mỗi trial chạy đủ năm fold qua common. Mục tiêu accuracy; không dùng test score. Tìm kiếm ngân sách nhỏ, không khẳng định tối ưu toàn cục. Giữ baseline khi trial không tốt hơn.

In [5]:
experiments.tuning(resume=not RECOMPUTE, n_trials=3)
display(pd.read_json(ROOT / 'evidence/models_D/selected_v1.json').T)
for family in ['lightgbm', 'xgboost', 'catboost', 'svm', 'mlp']:
    print(family)
    display(pd.read_csv(ROOT / f'evidence/models_D/{family}_trials.csv'))


[I 2026-11-13 22:45:32,604] Using an existing study with `study_name='lightgbm_v1_D'` instead of creating a new one.


Archived result: lightgbm_v1_trial00_D


Archived result: lightgbm_v1_trial01_D


Archived result: lightgbm_v1_trial02_D


SELECTED lightgbm: lightgbm_v1_trial00_D, accuracy=0.574673


[I 2026-11-13 22:45:33,046] Using an existing study with `study_name='xgboost_v1_D'` instead of creating a new one.


... (26 outputs left out)

svm


,number,value,datetime_start,datetime_complete,duration,params_C,params_gamma,state
0,0,0.542094,2026-11-13 22:09:49.683274,2026-11-13 22:11:41.198068,0 days 00:01:51.514794,5.0,0.003,COMPLETE
1,1,0.547096,2026-11-13 22:11:41.223168,2026-11-13 22:13:31.928603,0 days 00:01:50.705435,1.0,0.01,COMPLETE
2,2,0.527855,2026-11-13 22:13:31.953501,2026-11-13 22:16:47.133371,0 days 00:03:15.179870,20.0,scale,COMPLETE


mlp


,number,value,datetime_start,datetime_complete,duration,params_dropout,params_hidden_layers,params_learning_rate,params_weight_decay,state
0,0,0.533343,2026-11-13 22:16:47.388657,2026-11-13 22:16:55.076294,0 days 00:00:07.687637,0.00,64_32,0.002,0.001,COMPLETE
1,1,0.547304,2026-11-13 22:16:55.108227,2026-11-13 22:17:04.863821,0 days 00:00:09.755594,0.30,64,0.002,0.010,COMPLETE
2,2,0.547235,2026-11-13 22:17:04.888674,2026-11-13 22:17:20.230372,0 days 00:00:15.341698,0.15,64_32,0.001,0.000,COMPLETE


## 5. Đặc trưng v2 và nested OOF target encoding

Giữ siêu tham số chọn từ v1. Ba boosting được thử v2 thuần; bảy họ được thử v2 + TE. Encoder của C: smoothing 20, inner Stratified 4-fold seed 42 cho outer train; outer validation/test dùng mapping từ outer train. Nghệ sĩ lạ nhận prior. Không fit encoder trước khi chia outer CV.

In [6]:
experiments.v2(resume=not RECOMPUTE)
all_results = audit_module.audit_all()
assert len(all_results) == 32
display(all_results)


Archived result: lightgbm_v2_selected_D


Archived result: xgboost_v2_selected_D


Archived result: catboost_v2_selected_D


Already complete: lightgbm_v2_te_selected_D


Already complete: xgboost_v2_te_selected_D

Already complete: catboost_v2_te_selected_D

... (2 outputs left out)

Already complete: extratrees_v2_te_selected_D


Already complete: histgb_v2_te_selected_D


PASS: 32 models; probabilities, row counts, class order, all 5-fold metrics


,model,family,feature_version,target_encoding,cv_accuracy,cv_std,macro_f1,seconds,audit
0,catboost_v2_te_selected_D,catboost,v2,True,0.654765,0.003912,0.700522,86.076684,PASS
1,lightgbm_v2_te_selected_D,lightgbm,v2,True,0.654001,0.007181,0.706566,17.348645,PASS
2,xgboost_v2_te_selected_D,xgboost,v2,True,0.651084,0.003745,0.702729,34.235275,PASS
3,histgb_v2_te_selected_D,histgb,v2,True,0.641498,0.004274,0.692461,14.953408,PASS
4,mlp_v2_te_selected_D,mlp,v2,True,0.631495,0.005800,0.671531,19.392135,PASS
5,svm_v2_te_selected_D,svm,v2,True,0.629063,0.004937,0.665659,129.787016,PASS
6,extratrees_v2_te_selected_D,extratrees,v2,True,0.616351,0.005037,0.656789,18.352068,PASS
7,lightgbm_v2_selected_D,lightgbm,v2,False,0.576340,0.003094,0.618823,10.831525,PASS
8,xgboost_v1_trial00_D,xgboost,v1,False,0.575298,0.003518,0.614054,21.799891,PASS
9,xgboost_v1_base_D,xgboost,v1,False,0.575159,0.004627,0.615444,24.804181,PASS


## 6. Mục 5, Bảng 2 và bàn giao E

Kiểm lại xác suất, điểm từng fold, lớp 0–10, ID/thứ tự dòng. Bàn giao bảy họ khác nhau cùng manifest SHA-256. CV sau tuning dùng lại các folds lựa chọn, chưa phải ước lượng độc lập sau tuning. Không có điểm Kaggle mới. Phần E chịu trách nhiệm ensemble/phân tích lỗi.

In [7]:
selected = report.build_report()
display(selected)
display(pd.read_csv(ROOT / 'evidence/models_D/hyperparameters_D.csv'))
manifest = json.loads(manifest_path.read_text())
for item in manifest['models']:
    for relative, expected in item['files'].items():
        assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected
assert len(selected) >= 6
assert {'lightgbm','xgboost','catboost'}.issubset(set(selected.family))
assert {'svm','mlp'}.intersection(set(selected.family))
assert selected.iloc[0].cv_accuracy > 0.5585
print('PASS: 32 cấu hình, 7 họ bàn giao; đủ OOF/test/CV; vượt baseline; manifest đúng.')
display(Markdown((ROOT / 'handoff_D_to_E.md').read_text(encoding='utf-8')))


PASS: 32 models; probabilities, row counts, class order, all 5-fold metrics


Built paper section 5, table 2, handoff manifest for 7 families.


,model,family,feature_version,target_encoding,cv_accuracy,cv_std,macro_f1,seconds,audit
0,catboost_v2_te_selected_D,catboost,v2,True,0.654765,0.003912,0.700522,86.076684,PASS
1,lightgbm_v2_te_selected_D,lightgbm,v2,True,0.654001,0.007181,0.706566,17.348645,PASS
2,xgboost_v2_te_selected_D,xgboost,v2,True,0.651084,0.003745,0.702729,34.235275,PASS
3,histgb_v2_te_selected_D,histgb,v2,True,0.641498,0.004274,0.692461,14.953408,PASS
4,mlp_v2_te_selected_D,mlp,v2,True,0.631495,0.005800,0.671531,19.392135,PASS
5,svm_v2_te_selected_D,svm,v2,True,0.629063,0.004937,0.665659,129.787016,PASS
6,extratrees_v2_te_selected_D,extratrees,v2,True,0.616351,0.005037,0.656789,18.352068,PASS


,family,model,parameters,cv_accuracy
0,catboost,catboost_v2_te_selected_D,depth=4; learning_rate=0.1; l2_leaf_reg=20.0; ...,0.654765
1,lightgbm,lightgbm_v2_te_selected_D,num_leaves=15; learning_rate=0.025; n_estimato...,0.654001
2,xgboost,xgboost_v2_te_selected_D,max_depth=4; learning_rate=0.03; n_estimators=...,0.651084
3,histgb,histgb_v2_te_selected_D,max_iter=150; learning_rate=0.08; max_leaf_nod...,0.641498
4,mlp,mlp_v2_te_selected_D,"hidden_layers=[64, 32]; dropout=0.15; weight_d...",0.631495
5,svm,svm_v2_te_selected_D,C=1.0; gamma=0.01; probability=True,0.629063
6,extratrees,extratrees_v2_te_selected_D,n_estimators=400; min_samples_leaf=2,0.616351


PASS: 32 cấu hình, 7 họ bàn giao; đủ OOF/test/CV; vượt baseline; manifest đúng.


# Bàn giao D → E

Trần Đình Bảo Quốc — 3124411252. Dữ liệu và seed theo quy ước nhóm. Đã chạy xong baseline, boosting, tuning và v2 theo đúng thứ tự phân công. Bàn giao **7 họ mô hình**, gồm LightGBM, XGBoost, CatBoost, HistGB, MLP, SVM, ExtraTrees.

| Tag dùng với common.load_oof | Accuracy | Std (5 folds) | Macro F1 |
| --- | --- | --- | --- |
| `catboost_v2_te_selected_D` | 0.654765 | 0.003912 | 0.700522 |
| `lightgbm_v2_te_selected_D` | 0.654001 | 0.007181 | 0.706566 |
| `xgboost_v2_te_selected_D` | 0.651084 | 0.003745 | 0.702729 |
| `histgb_v2_te_selected_D` | 0.641498 | 0.004274 | 0.692461 |
| `mlp_v2_te_selected_D` | 0.631495 | 0.005800 | 0.671531 |
| `svm_v2_te_selected_D` | 0.629063 | 0.004937 | 0.665659 |
| `extratrees_v2_te_selected_D` | 0.616351 | 0.005037 | 0.656789 |

Mỗi tag có đủ `evidence/cv_logs/<tag>_oof.npy`, `<tag>_test.npy`, `<tag>_cv.json`. OOF (14396,11) giữ nguyên thứ tự train; test (3600,11) giữ nguyên thứ tự test; cột xác suất tương ứng `classes=[0,1,...,10]`. Test là trung bình xác suất năm outer models. Tất cả tag trên dùng v2 + TE và cùng outer folds seed 42. Không trộn thứ tự dòng hoặc tự mã hóa lại nhãn.

```python
import sys
sys.path.insert(0, 'code')  # chạy tại thư mục dự án
from common import load_oof
oof, test_proba = load_oof('catboost_v2_te_selected_D')
```

`evidence/models_D/` có:

- `selected_models_D.csv`: bảy mô hình bàn giao; `all_results_D.csv`: toàn bộ 32 cấu hình (có cả baseline và trial, không coi các trial là kết quả đánh giá độc lập).
- `y_encoded_D.npy`, `classes_D.npy`, `train_ids_D.npy`, `test_ids_D.npy`, `fold_ids_D.npy` (fold 0–4), để kiểm tra alignment trước ensemble.
- `handoff_manifest_D.json`: SHA-256 dữ liệu, mã nguồn, ba file của mỗi mô hình; phiên bản thư viện và thứ tự lớp.
- `hyperparameters_D.csv`, các `*_trials.csv`, `selected_v1.json`, `tuning.sqlite3` cho toàn bộ quá trình chọn cấu hình.
- `intermediate_D.zip`: các bộ kết quả không được chọn và đầu ra baseline tái lập. Các tệp này được nén gọn; notebook/audit đọc trực tiếp, không cần giải nén. Chỉ 21 tệp kết quả của bảy mô hình bàn giao được giữ rời để `common.load_oof` dùng ngay.

Mô hình đơn tốt nhất: **catboost_v2_te_selected_D, accuracy 0.654765**. Đây là CV sau lựa chọn trên cùng folds, chưa phải đánh giá độc lập sau tuning; không khẳng định ensemble sẽ cải thiện. Không có điểm leaderboard mới cho mô hình D. Không dùng public score để chọn. Tần suất train+test không nhãn được dùng theo quy ước nhóm; TE luôn trong fold và nested OOF ở phần train. So với baseline A 0,5585, mô hình tốt nhất tăng 9.63 điểm phần trăm.

Chạy tái lập: xem phần D trong README và `code/03_models_tuning_D.ipynb`. Mục 5 của paper: `paper/05_models_experimental_setup_D.md`; Bảng 2: `paper/table_02_hyperparameters_D.md`.

E cần tự kiểm chứng lựa chọn/trọng số ensemble bằng phương pháp tách OOF theo phân công. Không xem CV được tối ưu trên chính OOF là điểm đánh giá độc lập. Phần này là bàn giao file trong workspace, chưa gửi tin nhắn nhóm hoặc nộp Kaggle.
